In [1]:
import pandas as pd
import numpy as np
import os
import glob
from cleaning_script import adaptive_clean_real_estate_data
from feature_engineering import engineer_features

In [2]:
#GOING BACK AND LOOKING AT THE RAW DATASETS
data_dir = '../original_datasets'

# Pattern to match all CRMLS monthly files
pattern = os.path.join(data_dir, "CRMLSSold2025??_filled.csv")

# Get all matching file paths
file_list = sorted(glob.glob(pattern))

print("Files found:", file_list)

# Load and store each DataFrame
dfs = []
for f in file_list:
    df = pd.read_csv(f, low_memory=False)
    df["SourceFile"] = os.path.basename(f)  # optional: track the month
    dfs.append(df)

# Merge vertically (stack rows)
merged_df = pd.concat(dfs, ignore_index=True)

print("Merged shape:", merged_df.shape)



Files found: ['../original_datasets\\CRMLSSold202502_filled.csv', '../original_datasets\\CRMLSSold202503_filled.csv', '../original_datasets\\CRMLSSold202504_filled.csv', '../original_datasets\\CRMLSSold202505_filled.csv', '../original_datasets\\CRMLSSold202506_filled.csv', '../original_datasets\\CRMLSSold202507_filled.csv']
Merged shape: (133092, 81)


In [3]:
aug = pd.read_csv('../original_datasets/CRMLSSold202508_filled-2.csv')
sept = pd.read_csv('../original_datasets/CRMLSSold202509.csv')
merged_df

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,latfilled,lonfilled,SourceFile
0,RanchoSoutheast,RanchoSoutheast,NaN,True,NaN,NaN,NaN,60000.0,526199946,cmark1018@yahoo.com,...,False,NaN,NaN,92307,0.0,28000.0,NaN,False,False,CRMLSSold202502_filled.csv
1,InlandValleys,InlandValleys,NaN,False,NaN,NaN,NaN,550000.0,525585060,mozcorona@aol.com,...,False,NaN,NaN,92553,0.0,39640.0,NaN,False,False,CRMLSSold202502_filled.csv
2,SanDiego,SanDiego,NaN,False,NaN,NaN,False,880000.0,497696903,lenskab@gmail.com,...,False,2.0,NaN,91942,NaN,NaN,NaN,False,False,CRMLSSold202502_filled.csv
3,SanDiego,SanDiego,NaN,False,NaN,NaN,False,875000.0,497696407,lenskab@gmail.com,...,False,2.0,NaN,91942,NaN,NaN,NaN,False,False,CRMLSSold202502_filled.csv
4,SanDiego,SanDiego,NaN,False,NaN,NaN,False,849000.0,486616176,lenskab@gmail.com,...,False,2.0,NaN,91942,NaN,NaN,NaN,False,False,CRMLSSold202502_filled.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133087,OrangeCounty,OrangeCounty,NaN,True,NaN,NaN,NaN,7750000.0,1022700795,adam@OCcollective.com,...,False,NaN,NaN,92154,0.0,243065.0,NaN,False,False,CRMLSSold202507_filled.csv
133088,CaliforniaDesert,CaliforniaDesert,"Carpet,Tile",True,NaN,NaN,True,5800.0,1020000387,avi@unigrafix.com,...,False,2.0,NaN,92270,775.0,5227.0,NaN,False,False,CRMLSSold202507_filled.csv
133089,Glendale,Glendale,NaN,NaN,NaN,NaN,NaN,13850000.0,1019043037,Listings@LockerRealty.com,...,NaN,NaN,NaN,90277,NaN,68003.0,NaN,False,False,CRMLSSold202507_filled.csv
133090,OrangeCounty,OrangeCounty,NaN,True,NaN,NaN,NaN,2500000.0,1018313206,drew@teicheirateam.com,...,False,NaN,NaN,91730,0.0,1310720.0,NaN,False,False,CRMLSSold202507_filled.csv


In [4]:
merged_df = pd.concat([merged_df, aug, sept], ignore_index=True)
merged_df.shape

(178507, 81)

In [5]:
merged_df = merged_df[(merged_df['PropertyType']=='Residential') & (merged_df['PropertySubType']=='SingleFamilyResidence')]


In [6]:
testing_set = pd.read_csv('../original_datasets/CRMLSSold202510.csv')

In [7]:
train_cleaned, stats, mechs = adaptive_clean_real_estate_data(merged_df, fit = True)
test_clean, _, _ = adaptive_clean_real_estate_data(testing_set, fit = False, imputation_stats=stats, missing_mechanisms=mechs, trim_outliers=True )

ADAPTIVE CLEANING - TRAINING DATA
Initial shape: (89843, 81)

Dropping 21 columns with >75% missing

DIAGNOSING MISSING DATA MECHANISMS

ViewYN: NMAR (amenity - missing likely means absent)

PoolPrivateYN: NMAR (amenity - missing likely means absent)

AttachedGarageYN: NMAR (amenity - missing likely means absent)

LotSizeAcres: MAR detected (correlates with 1 features)
  - Latitude: r=-0.135


c:\Users\jueeh\idx-exchange-ds34\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\jueeh\idx-exchange-ds34\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\jueeh\idx-exchange-ds34\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\jueeh\idx-exchange-ds34\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]



BathroomsTotalInteger: MAR detected (correlates with 1 features)
  - LotSizeSquareFeet: r=0.104

FireplaceYN: NMAR (amenity - missing likely means absent)

Stories: MAR detected (correlates with 3 features)
  - Latitude: r=0.299
  - Longitude: r=-0.220
  - BathroomsTotalInteger: r=0.100

LotSizeArea: MAR detected (correlates with 1 features)
  - Latitude: r=-0.136

GarageSpaces: MAR detected (correlates with 1 features)
  - YearBuilt: r=-0.147

LotSizeSquareFeet: MAR detected (correlates with 2 features)
  - Latitude: r=-0.135
  - LotSizeAcres: r=0.871

--------------------------------------------------------------------------------
MECHANISM SUMMARY:
  MCAR: 10 columns
  MAR:  6 columns
  NMAR: 4 columns

ADAPTIVE IMPUTATION

ViewYN (NMAR): Created missing indicator, filled with False

PoolPrivateYN (NMAR): Created missing indicator, filled with False

Latitude (MCAR): Filled with 34.08

Longitude (MCAR): Filled with -118.03

LivingArea (MCAR): Filled with 1812.00

AttachedGarageYN (

c:\Users\jueeh\idx-exchange-ds34\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\jueeh\idx-exchange-ds34\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\jueeh\idx-exchange-ds34\xgboost_juee\cleaning_script.py:204: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False)
c:\Users\jueeh\idx-exchange-ds34\xgboost_juee\cleaning_script.py:204: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_opti


GarageSpaces (MAR): Will use model-based imputation

LotSizeSquareFeet (MAR): Will use model-based imputation

--------------------------------------------------------------------------------
Applying IterativeImputer to 6 MAR numeric features...
  ✓ Fitted IterativeImputer

⚠️  Remaining nulls: {'City': 57, 'PurchaseContractDate': 4, 'PostalCode': 1, 'SourceFile': 22908}

CLEANING COMPLETE
Final shape: (66871, 31)
Rows retained: 66871/89843 (74.4%)

Missingness indicators created: 4
  - ViewYN_was_missing: 9.2% of data
  - PoolPrivateYN_was_missing: 7.9% of data
  - AttachedGarageYN_was_missing: 11.9% of data
  - FireplaceYN_was_missing: 0.1% of data
ADAPTIVE CLEANING - TEST DATA
Initial shape: (23233, 78)

Dropping 20 columns with >75% missing

ADAPTIVE IMPUTATION

ViewYN (NMAR): Created missing indicator, filled with False

PoolPrivateYN (NMAR): Created missing indicator, filled with False

Latitude (MCAR): Filled with 34.08

Longitude (MCAR): Filled with -118.03

LivingArea (MCAR)

c:\Users\jueeh\idx-exchange-ds34\xgboost_juee\cleaning_script.py:204: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False)
c:\Users\jueeh\idx-exchange-ds34\xgboost_juee\cleaning_script.py:204: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False)
c:\Users\jueeh\idx-exchange-ds34\xgboost_juee\cleaning_script.py:204: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the futur

GOAL: BRAINSTORM NEIGHBORHOOD LEVEL FEATURES

In [8]:
train_cleaned.columns
location_cols = ['Latitude', 'Longitude', 'CountyOrParish', 'City', 'PostalCode']

In [9]:
test_clean = test_clean.drop(columns=['ClosePrice'])

In [10]:
train_cleaned

,ViewYN,PoolPrivateYN,CloseDate,ClosePrice,Latitude,Longitude,LivingArea,CountyOrParish,AttachedGarageYN,ParkingTotal,...,NewConstructionYN,GarageSpaces,PostalCode,LotSizeSquareFeet,SourceFile,ViewYN_was_missing,PoolPrivateYN_was_missing,AttachedGarageYN_was_missing,FireplaceYN_was_missing,PropertyAgeAtClose
2,False,False,2025-02-11,875000.0,32.765380,-117.043486,2340.0,San Diego,True,6.0,...,False,2.000000,91942,450228.840769,CRMLSSold202502_filled.csv,0,0,0,0,4.0
3,False,False,2025-02-14,875000.0,32.765038,-117.043568,2165.0,San Diego,True,4.0,...,False,2.000000,91942,446501.144831,CRMLSSold202502_filled.csv,0,0,0,0,4.0
4,False,False,2025-02-18,849000.0,32.765031,-117.043252,2158.0,San Diego,True,4.0,...,False,2.000000,91942,446442.624247,CRMLSSold202502_filled.csv,0,0,0,0,4.0
15,True,False,2025-02-28,1100000.0,33.724000,-118.294924,2545.0,Los Angeles,False,3.0,...,False,2.000000,90731,4501.000000,CRMLSSold202502_filled.csv,0,0,0,0,63.0
16,False,False,2025-02-28,760000.0,37.687556,-122.150114,1692.0,Alameda,True,2.0,...,False,2.000000,94579,5040.000000,CRMLSSold202502_filled.csv,1,0,0,0,75.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133047,True,True,2025-07-02,2625000.0,33.602236,-116.147017,7513.0,Riverside,True,10.0,...,False,10.000000,92274,13939.000000,CRMLSSold202507_filled.csv,0,0,0,0,6.0
133062,True,False,2025-07-25,375000.0,34.118035,-116.428352,1770.0,San Bernardino,True,2.0,...,True,2.000000,92284,4146.000000,CRMLSSold202507_filled.csv,0,0,0,0,3.0
133068,True,False,2025-07-03,1980000.0,39.357862,-123.821861,4700.0,Mendocino,False,6.0,...,False,2.000000,95460,24393.600000,CRMLSSold202507_filled.csv,0,0,0,0,46.0
133073,True,False,2025-07-19,550000.0,34.481528,-117.266840,3045.0,San Bernardino,True,2.0,...,False,2.000000,92395,7200.000000,CRMLSSold202507_filled.csv,0,0,0,0,35.0


In [11]:
train_with_features, test_with_features, metadata = engineer_features(train_cleaned, test_clean)

c:\Users\jueeh\idx-exchange-ds34\xgboost_juee\feature_engineering.py:316: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test[col + "_MedianPrice"].fillna(global_price_med, inplace=True)
c:\Users\jueeh\idx-exchange-ds34\xgboost_juee\feature_engineering.py:317: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values alw

KeyError: "['ClosePrice'] not in index"